In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

# 1. Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # make it (N, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)


In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)



In [ ]:
# 3. Create DataLoaders


# 3. Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [ ]:
# 4. Print shape of one batch

for images, labels in train_loader:
    print(f"Batch images shape: {images.shape}")  # should be (32, C, H, W)
    print(f"Batch labels shape: {labels.shape}")  # should be (32, 1)
    break


In [ ]:
# 5. Display sample images

images, labels = next(iter(train_loader))
images = images.numpy()

plt.figure(figsize=(12, 6))
for i in range(6):
    img = np.transpose(images[i], (1, 2, 0))  # convert back to HWC for matplotlib
    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(f"Age: {labels[i].item():.0f}")
    plt.axis('off')
plt.show()


In [ ]:
# Task 1: Write your model class here:

import torch
import torch.nn as nn

class AgePredictor(nn.Module):
    def __init__(self, input_size):
        super(AgePredictor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1)  # regression output
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
# Task 2: Write your training loop here:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Flattened image size
C, H, W = X_train.shape[1:]
input_size = C * H * W

model = AgePredictor(input_size).to(device)
criterion = nn.MSELoss()  # regression loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 3: Write your validation loop here:

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # Flatten images
        images = images.view(images.size(0), -1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


In [ ]:
# Task 4: Define device, model, loss, optimizer:

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            images = images.view(images.size(0), -1)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


In [ ]:
# Task 5: Start training for 20 epochs:

num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")


In [ ]:
# Task 1: Write your code here:

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(range(1, num_epochs+1), train_losses, label='Training Loss', marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label='Validation Loss', marker='o')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:

# Get a batch from test_loader
model.eval()
images, labels = next(iter(test_loader))

# Move to device and flatten
images_device = images.to(device)
outputs = model(images_device.view(images_device.size(0), -1))
preds = outputs.cpu().detach().numpy()
labels = labels.numpy()
images = images.numpy()

# Display first 6 images with predicted vs actual ages
plt.figure(figsize=(12,6))
for i in range(6):
    img = np.transpose(images[i], (1,2,0))  # C,H,W -> H,W,C
    plt.subplot(2,3,i+1)
    plt.imshow(img)
    plt.title(f"Pred: {preds[i][0]:.1f}, Actual: {labels[i][0]:.0f}")
    plt.axis('off')
plt.show()
